# 1. Economic data audit

What was fetched, against what the registry declared, and how much it has been
revised since. This is stage gate one: nothing downstream is worth reading until
every series fetches, caches, and reloads identically.

Run `forecast fetch-data` before this notebook. All computation lives in the
package; this notebook only displays.

In [ ]:
from datetime import date

import pandas as pd

from economic_regime_forecasting import pipeline_gates
from economic_regime_forecasting.configuration.registry import load_registries
from economic_regime_forecasting.configuration.run_settings import DEFAULT_RUN_SETTINGS
from economic_regime_forecasting.data import audit, indicator_outcomes
from economic_regime_forecasting.data.cache import SeriesCache
from economic_regime_forecasting.data.panel import assemble_point_in_time_panel, load_final_series
from economic_regime_forecasting.features.observation_matrix import build_observation_matrix
from economic_regime_forecasting.reporting import figures

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

settings = DEFAULT_RUN_SETTINGS
registry, indicators = load_registries()
cache = SeriesCache(settings.cache.raw, settings.cache.vintage)
today = date.today()
print(f"{len(registry.series)} series, {len(indicators)} indicators, as of {today}")

## Every series, measured rather than assumed

The registry declares a start date for each series. This table measures the
actual one and flags any disagreement, so a declared fact cannot quietly go stale.

In [ ]:
series_audits = audit.audit_series(registry, cache)
audit_table = audit.audit_table(series_audits)
audit_table

In [ ]:
final_series = load_final_series(registry, cache, [item.name for item in registry.series])
figures.plot_series_panel(final_series)

## How much do these numbers move after publication?

The comparison is on year-over-year growth rather than the raw index, because the
index has been rebased repeatedly and a rebasing is not a revision. A 1970 vintage
puts January 1919 industrial production at 24.6 where a 2026 vintage puts it at
4.87; in growth terms those two vintages agree.

The `policy` column says which of the three point-in-time policies each vintage
date used. `publication_lag_fallback` means the archive had nothing usable that
far back, so the timing is right and the values are revised.

In [ ]:
revisions = audit.audit_revisions(
    registry, cache, [date(year, 1, 1) for year in (1975, 1990, 2005, 2020)]
)
audit.revision_table(revisions)

## The panel the model actually reads

Three standardised series, growth then inflation then rates. Its start is set by
the shortest input after a twelve-month growth window and a thirty-six month
standardisation window.

In [ ]:
matrix = build_observation_matrix(assemble_point_in_time_panel(registry, today, cache), registry)
print(matrix.describe())
if matrix.bridged_months:
    print("interpolated across:", [f"{m:%Y-%m}" for m in matrix.bridged_months])
matrix.standardised.describe().round(3)

## What the indicators resolve to

Each indicator's unconditional base rate over everything history has resolved.
These are the numbers the model has to beat.

In [ ]:
outcome_series = load_final_series(
    registry, cache, indicator_outcomes.required_series_names(indicators)
)
resolved = indicator_outcomes.resolve_all(
    indicators, outcome_series, settings.forecast_horizons_in_months
)
pd.DataFrame(
    [
        {
            "indicator": name,
            "horizon_years": horizon // 12,
            "resolved": item.resolved_count,
            "positives": item.positive_count,
            "base_rate": round(item.base_rate, 3),
        }
        for (name, horizon), item in resolved.items()
    ]
).pivot(index="indicator", columns="horizon_years", values="base_rate")

## Gate 1

In [ ]:
reload_cache = SeriesCache(settings.cache.raw, settings.cache.vintage)
audit.audit_series(registry, reload_cache)
print(pipeline_gates.gate_one_data(registry, series_audits, reload_cache.statistics).describe())